In [ ]:
# !pip install xarray rioxarray netCDF4 h5netcdf rasterio scipy
# !pip install netCDF4

Yang dilakukan:
1. Membangun arsitektur ConvLSTM2D menggunakan TensorFlow
2. Memuat data .npy yang dihasilkan dari tahap preprocessing
3. Menjalankan proses pelatihan (training) dengan fungsi Weighted Binary Cross-Entropy Loss
4. Melakukan validasi (menghitung Recall/Precision) dan menyimpan model terbaik menjadi file .h5 yang nantinya akan dipakai di Website/Backend

In [39]:
import os
import pandas as pd
import numpy as np
import glob
import h5py
from netCDF4 import Dataset
from datetime import datetime

In [40]:
LST_FOLDER = "../data/raw/lst"

kalimantan_bbox = {
    "min_lon": 108.0, "max_lon": 118.5,
    "min_lat": -4.5,  "max_lat": 3.0
}

TARGET_TILE = "h29v09"

In [ ]:
all_files = glob.glob("../data/raw/fire_nrt_*.csv")

if not all_files:
    print("❌ Error: Tidak ada file CSV ditemukan!")
else:
    print(f"📂 Ditemukan {len(all_files)} file CSV. Sedang menggabungkan...")
    
    li = []
    for filename in all_files:
        df = pd.read_csv(filename, index_col=None, header=0)
        li.append(df)

    df_all = pd.concat(li, axis=0, ignore_index=True)

    initial_len = len(df_all)
    df_all.drop_duplicates(subset=['latitude', 'longitude', 'acq_date', 'acq_time', 'satellite'], inplace=True)
    print(f"🧹 Menghapus {initial_len - len(df_all)} data duplikat.")

    df_kal = df_all[
        (df_all["longitude"] >= kalimantan_bbox["min_lon"]) &
        (df_all["longitude"] <= kalimantan_bbox["max_lon"]) &
        (df_all["latitude"] >= kalimantan_bbox["min_lat"]) &
        (df_all["latitude"] <= kalimantan_bbox["max_lat"])
    ].copy()

    df_kal['acq_date'] = pd.to_datetime(df_kal['acq_date']).dt.strftime('%Y-%m-%d')

    print(f"✅ Total Data Final: {len(df_kal)} baris titik api.")

📂 Ditemukan 3 file CSV. Sedang menggabungkan...
🧹 Menghapus 0 data duplikat.
✅ Total Data Final: 200906 baris titik api.


In [42]:
def date_to_julian(date_str):
    """
    Mengubah tanggal biasa '2024-01-01' menjadi format Julian MODIS 'A2024001'.
    """
    try:
        dt = datetime.strptime(str(date_str), "%Y-%m-%d")
        return f"A{dt.year}{dt.timetuple().tm_yday:03d}"
    except:
        return None

In [ ]:
def get_daily_lst_regional(date_obj):
    """
    Membaca file MODIS LST asli.
    Mengembalikan rata-rata suhu tanah (Celcius).
    """
    try:
        if isinstance(date_obj, str):
            date_obj = pd.to_datetime(date_obj)
        julian_str = f"A{date_obj.year}{date_obj.dayofyear:03d}"
        
        search_pattern = os.path.join(LST_FOLDER, f"*{julian_str}*{TARGET_TILE}*.hdf")
        found_files = glob.glob(search_pattern)
        
        if not found_files:
            return np.nan 
            
        file_path = found_files[0]
        
        with Dataset(file_path, mode='r') as ds:
            var_name = None
            for key in ds.variables.keys():
                if "LST_Day_1km" in key:
                    var_name = key
                    break
            
            if var_name:
                data = ds.variables[var_name][:]
                
                valid_data = data[data > 0] 
                
                if len(valid_data) > 0:
                    avg_kelvin = np.mean(valid_data)
                    
                    if avg_kelvin > 1000: 
                        avg_kelvin = avg_kelvin * 0.02

                    return avg_kelvin - 273.15
                    
    except Exception as e:
        return np.nan
        
    return np.nan

In [ ]:
unique_dates = df_kal['acq_date'].unique()
print(f"   -> Mencari data untuk {len(unique_dates)} hari unik...")

daily_temps = {}
for d in unique_dates:
    daily_temps[d] = get_daily_lst_regional(d)

df_kal['lst_regional_mean'] = df_kal['acq_date'].map(daily_temps)

filled = df_kal['lst_regional_mean'].count()
total = len(df_kal)
print(f"📊 Status LST HDF: Terisi {filled} dari {total} data.")

if filled < total:
    missing_mask = df_kal['lst_regional_mean'].isna()
    print(f"⚠️ Ada {missing_mask.sum()} data kosong (File HDF tidak ada).")
    print("   -> Mengisi kekosongan dengan rata-rata suhu yang ada (Imputasi).")
    
    mean_lst = df_kal['lst_regional_mean'].mean()
    if np.isnan(mean_lst): mean_lst = 30.0 
    
    df_kal['lst_regional_mean'] = df_kal['lst_regional_mean'].fillna(mean_lst)

print(f"✅ FINAL LST: Semua terisi.")
print(f"   Contoh Data (Celcius): {df_kal['lst_regional_mean'].head(3).values}")

   -> Mencari data untuk 358 hari unik...
📊 Status LST HDF: Terisi 200901 dari 200906 data.
⚠️ Ada 5 data kosong (File HDF tidak ada).
   -> Mengisi kekosongan dengan rata-rata suhu yang ada (Imputasi).
✅ FINAL LST: Semua terisi.
   Contoh Data (Celcius): [25.88297066 25.88297066 25.88297066]


In [ ]:
GPM_FOLDER = "../data/raw/rainfall"

MIN_LON, MAX_LON = 108.0, 118.5
MIN_LAT, MAX_LAT = -4.5, 3.0

In [ ]:
def get_regional_rainfall(date_input):
    if isinstance(date_input, pd.Timestamp):
        date_str = date_input.strftime('%Y-%m-%d')
    else:
        date_str = str(date_input)

    filename = f"gpm_{date_str}.nc4"
    file_path = os.path.join(GPM_FOLDER, filename)
    
    if not os.path.exists(file_path):
        return np.nan 

    try:
        ds = Dataset(file_path, mode='r')
        
        if 'Grid' in ds.groups:
            data_grp = ds.groups['Grid']
        else:
            data_grp = ds 
            
        if 'precipitation' not in data_grp.variables:
            ds.close()
            return np.nan

        precip_var = data_grp.variables['precipitation']
        lats = data_grp.variables['lat'][:]
        lons = data_grp.variables['lon'][:]

        lat_idx = np.where((lats >= MIN_LAT) & (lats <= MAX_LAT))[0]
        lon_idx = np.where((lons >= MIN_LON) & (lons <= MAX_LON))[0]
        
        if len(lat_idx) == 0 or len(lon_idx) == 0:
            ds.close()
            return np.nan
        
        if precip_var.shape[1] == len(lons):
            data_clip = precip_var[0, lon_idx.min():lon_idx.max()+1, lat_idx.min():lat_idx.max()+1]
            
            data_clip = data_clip.T 
        else:
            data_clip = precip_var[0, lat_idx.min():lat_idx.max()+1, lon_idx.min():lon_idx.max()+1]
        
        valid_rain = data_clip[(data_clip >= 0) & (data_clip < 10000)]
        
        if len(valid_rain) > 0:
            avg_rain = np.mean(valid_rain)
            ds.close()
            return avg_rain
            
        ds.close()
        
    except Exception as e:
        pass
        
    return np.nan

In [ ]:
print("⏳ Sedang mengekstrak ulang data Curah Hujan (GPM)...")

unique_dates = df_kal['acq_date'].unique()

daily_rain_dict = {}
for d in unique_dates:
    daily_rain_dict[d] = get_regional_rainfall(d)

df_kal["rainfall_regional_mean"] = df_kal["acq_date"].map(daily_rain_dict)

filled_gpm = df_kal["rainfall_regional_mean"].notna().sum()

print(f"\n✅ SELESAI!")
print(f"📊 Statistik GPM:")
print(f"   - Data Hujan Terisi: {filled_gpm} dari {len(df_kal)}")

if filled_gpm > 0:
    print("\n🔍 Sampel Data:")
    print(df_kal[df_kal["rainfall_regional_mean"].notna()][["acq_date", "rainfall_regional_mean"]].head())

⏳ Sedang mengekstrak ulang data Curah Hujan (GPM)...

✅ SELESAI!
📊 Statistik GPM:
   - Data Hujan Terisi: 200906 dari 200906

🔍 Sampel Data:
     acq_date  rainfall_regional_mean
0  2023-01-01                1.286755
1  2023-01-01                1.286755
2  2023-01-01                1.286755
3  2023-01-01                1.286755
4  2023-01-01                1.286755


In [ ]:
VEG_FOLDER = "../data/raw/vegetation"

def read_ndvi_from_file(filepath):
    try:
        with Dataset(filepath, mode='r') as ds:
            ds.set_auto_maskandscale(False)
            
            var_name = None
            for key in ds.variables.keys():
                if "NDVI" in key:
                    var_name = key
                    break
            
            if var_name:
                data = ds.variables[var_name][:]
                valid_data = data[(data >= -2000) & (data <= 10000)]
                
                if len(valid_data) > 0:
                    return np.mean(valid_data) * 0.0001
    except:
        return np.nan
    return np.nan

In [ ]:
all_files = glob.glob(os.path.join(VEG_FOLDER, f"*{TARGET_TILE}*.hdf"))
print(f"   -> Ditemukan {len(all_files)} file NDVI mentah.")

ndvi_map = {} 

for f in all_files:
    try:
        basename = os.path.basename(f)
        parts = basename.split('.')
        for p in parts:
            if p.startswith('A2023') and len(p) == 8:
                dt = pd.to_datetime(p[1:], format='%Y%j')
                
                val = read_ndvi_from_file(f)
                if not np.isnan(val):
                    ndvi_map[dt] = val
                break
    except:
        continue

print(f"   -> Berhasil mengekstrak {len(ndvi_map)} titik data waktu.")

   -> Ditemukan 24 file NDVI mentah.
   -> Berhasil mengekstrak 23 titik data waktu.


In [ ]:
df_kal['temp_date'] = pd.to_datetime(df_kal['acq_date'])
df_kal['ndvi_regional'] = df_kal['temp_date'].map(ndvi_map)

print("   -> Melakukan Interpolasi Linear untuk mengisi gap 16 hari...")
df_kal = df_kal.sort_values('acq_date') 
df_kal['ndvi_regional'] = df_kal['ndvi_regional'].interpolate(method='linear', limit_direction='both')

df_kal.drop(columns=['temp_date'], inplace=True)

filled = df_kal['ndvi_regional'].count()
total = len(df_kal)
print(f"✅ FINAL NDVI: Terisi {filled} dari {total}")
print(f"   Contoh Data: {df_kal['ndvi_regional'].head(3).values}")

if filled == 0:
    print("⚠️ Data kosong total. Mengisi default hutan tropis (0.6).")
    df_kal['ndvi_regional'] = 0.6

   -> Melakukan Interpolasi Linear untuk mengisi gap 16 hari...
✅ FINAL NDVI: Terisi 200906 dari 200906
   Contoh Data: [0.67049393 0.67049393 0.67049393]


In [ ]:
print("💾 MENYIMPAN DATASET FINAL...")
output_csv = "../data/processed/dataset_FireFinder_ai_kalimantan.csv"
df_kal.to_csv(output_csv, index=False)
print(f"🎉 SUKSES BESAR! File disimpan: {output_csv}")
print(f"   Siap untuk Modeling AI! 🚀")

💾 MENYIMPAN DATASET FINAL...
🎉 SUKSES BESAR! File disimpan: ../data/processed/dataset_FireFinder_ai_kalimantan.csv
   Siap untuk Modeling AI! 🚀
